<a href="https://colab.research.google.com/github/Krishnap641/ABC_Fin_App/blob/master/RAG_Pipeline_from_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sqlalchemy pandas faiss-cpu sentence-transformers transformers accelerate torch tiktoken spacy -q
!python -m spacy download en_core_web_sm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 50.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 44.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import sqlite3

# Connect to SQLite database (creates the file)
conn = sqlite3.connect('example.db')
cursor = conn.cursor()

# Create the 'notes' table
cursor.execute('''
CREATE TABLE IF NOT EXISTS notes (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    content TEXT NOT NULL,
    author TEXT
)
''')

# Insert some documents into the table
notes_data = [
    (1, 'Project Gemini', 'Project Gemini was NASA\'s second human spaceflight program. It was a key step towards the Apollo moon landing. Its objective was to develop space travel techniques for the Apollo mission.', 'Admin'),
    (2, 'Database Indexing', 'An index is a data structure that improves the speed of data retrieval on a database table. Indexes are used to quickly locate data without searching every row.', 'DB_Guru'),
    (3, 'Renewable Energy', 'Solar power harnesses energy from the sun using photovoltaic panels. Wind power uses turbines to convert wind into electricity.', 'EcoWarrior'),
    (4, 'The FAISS Library', 'FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors. It is written in C++ with a Python wrapper.', 'AI_Researcher')
]
cursor.executemany('INSERT INTO notes (id, title, content, author) VALUES (?, ?, ?, ?)', notes_data)

conn.commit()
conn.close()

print("✅ Database 'example.db' created successfully.")

✅ Database 'example.db' created successfully.


In [3]:
import os
import re
import pickle
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import spacy
import tiktoken

# Load spacy model
nlp = spacy.load("en_core_web_sm")

# ----------------------------- Utilities & Chunking -----------------------------
def fetch_table(engine_url: str, table: str, cols: List[str], id_col: str) -> pd.DataFrame:
    engine = create_engine(engine_url)
    with engine.connect() as conn:
        all_cols = ", ".join([f'"{c}"' for c in cols + [id_col]])
        df = pd.read_sql_query(text(f'SELECT {all_cols} FROM "{table}"'), conn)
    return df

def chunk_text(text: str, max_tokens: int = 500, overlap: int = 50) -> List[str]:
    enc = tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]
        chunks.append(enc.decode(chunk_tokens))
        if end == len(tokens):
            break
        start += max_tokens - overlap
    return chunks

# ----------------------------- Embeddings & Indexing -----------------------------
class EmbeddingModel:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed_texts(self, texts: List[str]) -> np.ndarray:
        return self.model.encode(texts, convert_to_numpy=True).astype(np.float32)

def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    dim = embeddings.shape[1]
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(dim)
    # Use GPU if available
    try:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
        print("FAISS index moved to GPU.")
    except Exception:
        print("GPU for FAISS not available, using CPU.")
    index.add(embeddings)
    return index

# ----------------------------- Main Pipeline -----------------------------
def pipeline_from_sql(engine_url: str, table: str, text_cols: List[str], id_col: str, persist_dir: str = "./rag_index"):
    os.makedirs(persist_dir, exist_ok=True)
    df = fetch_table(engine_url, table, text_cols, id_col)
    docs = []
    for row in df.to_dict(orient="records"):
        combined_text = "\n\n".join([str(row.get(c, "")) for c in text_cols])
        if not combined_text.strip():
            continue
        chunks = chunk_text(combined_text)
        for i, chunk in enumerate(chunks):
            doc_id = f"{row.get(id_col)}_{i}"
            docs.append({"id": doc_id, "text": chunk, "source_id": row.get(id_col)})

    texts = [d["text"] for d in docs]
    embedding_model = EmbeddingModel()
    embeddings = embedding_model.embed_texts(texts)
    index = build_faiss_index(embeddings)

    # Move index back to CPU for saving
    if faiss.get_num_gpus() > 0:
        index = faiss.index_gpu_to_cpu(index)

    faiss.write_index(index, os.path.join(persist_dir, "faiss.index"))
    with open(os.path.join(persist_dir, "metadata.pkl"), "wb") as f:
        pickle.dump(docs, f)

    print(f"✅ Indexed {len(docs)} chunks. Index and metadata saved to {persist_dir}")

# ----------------------------- Retrieval & Generation -----------------------------
def load_index_and_meta(persist_dir: str = "./rag_index") -> Tuple[faiss.Index, List[Dict]]:
    index = faiss.read_index(os.path.join(persist_dir, "faiss.index"))
    try:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
        print("FAISS index loaded to GPU.")
    except Exception:
        print("FAISS index loaded to CPU.")
    with open(os.path.join(persist_dir, "metadata.pkl"), "rb") as f:
        docs = pickle.load(f)
    return index, docs

def search(query: str, embedding_model: EmbeddingModel, index: faiss.Index, docs: List[Dict], k: int = 3) -> List[Dict]:
    q_emb = embedding_model.embed_texts([query])
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            results.append({**docs[idx], "score": float(score)})
    return results

def generate_answer(query: str, retrieved_docs: List[Dict]):
    context = "\n\n---\n\n".join([f"Source ID {d['source_id']}:\n{d['text']}" for d in retrieved_docs])
    model_name = "microsoft/phi-2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", trust_remote_code=True).to("cuda")

    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=150, device=0)
    prompt = f"Using only the context provided, answer the question.\n\nContext:\n{context}\n\nQuestion:\n{query}\nAnswer:"

    output = pipe(prompt, do_sample=False)[0]["generated_text"]
    return output[len(prompt):].strip()

print("✅ RAG script functions loaded.")

✅ RAG script functions loaded.


In [4]:
pipeline_from_sql(
    engine_url="sqlite:///./example.db",
    table="notes",
    text_cols=["title", "content"],
    id_col="id"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

GPU for FAISS not available, using CPU.
✅ Indexed 4 chunks. Index and metadata saved to ./rag_index


In [5]:
# Create a single instance of the embedding model to use for searching
emb_model = EmbeddingModel()

# Load the index and metadata we just created
index, docs = load_index_and_meta()

# Define your question
my_query = "What was the purpose of Project Gemini?"

# 1. Retrieve relevant documents
retrieved_docs = search(my_query, emb_model, index, docs)

print("🔍 Top Retrieved Documents:\n")
for doc in retrieved_docs:
    print(f"  Score: {doc['score']:.4f}, Source ID: {doc['source_id']}")
    print(f"  Text: {doc['text'][:200]}...\n")

# 2. Generate an answer based on the retrieved documents
print("\n💬 Generating Answer...\n")
answer = generate_answer(my_query, retrieved_docs)

print(f"Answer: {answer}")

FAISS index loaded to CPU.
🔍 Top Retrieved Documents:

  Score: 0.8205, Source ID: 1
  Text: Project Gemini

Project Gemini was NASA's second human spaceflight program. It was a key step towards the Apollo moon landing. Its objective was to develop space travel techniques for the Apollo missi...

  Score: 0.1024, Source ID: 3
  Text: Renewable Energy

Solar power harnesses energy from the sun using photovoltaic panels. Wind power uses turbines to convert wind into electricity....

  Score: 0.0651, Source ID: 4
  Text: The FAISS Library

FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors. It is written in C++ with a Python wrapper....


💬 Generating Answer...



tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

... (7 outputs left out)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer: The purpose of Project Gemini was to develop space travel techniques for the Apollo mission.
